In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd "/content/drive/MyDrive/Courses/AI/masked_attention/llama_like/excercises"

/content/drive/MyDrive/Courses/AI/masked_attention/llama_like/excercises


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/Courses/AI/masked_attention/llama_like")

In [ ]:
"""
Taller: Internals de un LLM estilo LLaMA
=================================================================
Instrucciones:
  - Busca los bloques marcados con TODO y completa el código.
  - Cada sección tiene una celda de verificación al final.
  - No modifiques nada fuera de los bloques TODO.

Secciones:
   4. RoPE ablation — qué pasa sin codificación posicional
"""

'\nTaller: Internals de un LLM estilo LLaMA\n=================================================================\nInstrucciones:\n  - Busca los bloques marcados con TODO y completa el código.\n  - Cada sección tiene una celda de verificación al final.\n  - No modifiques nada fuera de los bloques TODO.\n \nSecciones:\n  1. RMSNorm y SwiGLU desde cero\n'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

# Importamos el modelo de referencia para comparaciones
from src.model import (
    ModelConfig, RMSNorm, SwiGLUFFN, MiniLLaMA,
    precompute_rope_freqs, apply_rope, GroupedQueryAttention
)
from src.data import get_corpus
from src.tokenizer import BPETokenizer

In [ ]:
# ===========================================================================
# SECCIÓN 4 — RoPE ablation
# ===========================================================================
# ¿Qué pasa si entrenamos sin codificación posicional?
# Compara la curva de loss de un modelo con RoPE vs uno sin RoPE.
#
# Para quitar RoPE, basta con no rotar Q y K antes del dot product.
# ===========================================================================

class AttentionNoRoPE(nn.Module):
    """GroupedQueryAttention sin rotación posicional."""

    def __init__(self, config: ModelConfig):
        super().__init__()
        assert config.n_heads % config.n_kv_heads == 0
        self.n_heads    = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.n_rep      = config.n_heads // config.n_kv_heads
        self.head_dim   = config.d_model // config.n_heads
        D, Dh           = config.d_model, self.head_dim

        self.Wq = nn.Linear(D, config.n_heads    * Dh, bias=False)
        self.Wk = nn.Linear(D, config.n_kv_heads * Dh, bias=False)
        self.Wv = nn.Linear(D, config.n_kv_heads * Dh, bias=False)
        self.Wo = nn.Linear(config.n_heads * Dh, D,    bias=False)

    def forward(self, x, rope_freqs, mask=None):
        B, T, D = x.shape
        Dh      = self.head_dim

        q = self.Wq(x).reshape(B, T, self.n_heads,    Dh)
        k = self.Wk(x).reshape(B, T, self.n_kv_heads, Dh)
        v = self.Wv(x).reshape(B, T, self.n_kv_heads, Dh)

        # TODO 4.1 — Esta versión NO debe aplicar RoPE a Q y K
        #            Comenta o elimina las líneas de apply_rope
        #            (en el módulo original están aquí)
        raise NotImplementedError("TODO 4.1")

        k = k.unsqueeze(3).expand(B, T, self.n_kv_heads, self.n_rep, Dh)\
             .reshape(B, T, self.n_heads, Dh)
        v = v.unsqueeze(3).expand(B, T, self.n_kv_heads, self.n_rep, Dh)\
             .reshape(B, T, self.n_heads, Dh)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scale  = math.sqrt(Dh)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out  = torch.matmul(attn, v)
        out  = out.transpose(1, 2).reshape(B, T, self.n_heads * Dh)
        return self.Wo(out)


def train_quick(use_rope: bool, epochs: int = 100) -> list[float]:
    """Entrena MiniLLaMA con o sin RoPE y retorna la curva de loss."""
    # TODO 4.2 — Completa esta función:
    #   1. Carga el corpus y el tokenizer (usa BPETokenizer().load(...))
    #   2. Si use_rope=False, reemplaza model.layer.attn por AttentionNoRoPE(cfg)
    #   3. Entrena por `epochs` epochs con AdamW lr=3e-3
    #   4. Retorna lista de loss promedio por epoch
    # return losses
    raise NotImplementedError("TODO 4.2")


# ── Verificación 4 ──────────────────────────────────────────────────────────
def verify_section4():
    print("=" * 55)
    print("VERIFICACIÓN 4 — RoPE ablation  (entrena ~100 epochs)")
    print("=" * 55)

    losses_rope   = train_quick(use_rope=True,  epochs=100)
    losses_no_rope = train_quick(use_rope=False, epochs=100)

    final_rope    = losses_rope[-1]
    final_no_rope = losses_no_rope[-1]
    print(f"  Loss final con RoPE    : {final_rope:.4f}")
    print(f"  Loss final sin RoPE    : {final_no_rope:.4f}")
    print(f"  RoPE mejora el loss    : {final_rope < final_no_rope}")

    plt.figure(figsize=(7, 4))
    plt.plot(losses_rope,    label="con RoPE",    linewidth=1.5)
    plt.plot(losses_no_rope, label="sin RoPE",    linewidth=1.5, linestyle="--")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("RoPE ablation — curva de entrenamiento")
    plt.legend()
    plt.tight_layout()
    plt.savefig("rope_ablation.png", dpi=130)
    print("  Plot guardado en rope_ablation.png")

    if final_rope < final_no_rope:
        print("\n  ✓ Sección 4 correcta")
    else:
        print("\n  ~ Resultado inesperado — revisa train_quick()")

In [ ]:
verify_section4()